In [5]:
import ast
import re
from collections import Counter
from fractions import Fraction

In [22]:
ANSWER_RE = re.compile(r"<answer>(.*?)</answer>", re.DOTALL | re.IGNORECASE)


def is_conversational(messages):
    if isinstance(messages, list):
        message = messages[0]
        # Each message must a list of dictionaries with keys "role" and "content"
        if isinstance(message, dict) and "role" in message and "content" in message:
            return True
    return False


def normalize_ops(s: str) -> str:
    """Normalize common unicode operator variants to ASCII."""
    return s.replace("×", "*").replace("÷", "/").replace("−", "-").strip()


def extract_final_answer_text(completion: str) -> str | None:
    """Return the inner string of the (single) <answer>...</answer>.
    Returns None if missing OR if there is not exactly one answer block.
    Adds synthetic <think> prefix to match your current behavior.
    """
    completion = "<think>" + completion
    matches = ANSWER_RE.findall(completion or "")
    if len(matches) == 0:
        return None
    ans = normalize_ops(matches[-1])
    return ans if ans else None


def split_lhs_rhs(answer_text: str) -> tuple[str, str | None]:
    """If '=' is present, treat as equation and split on the LAST '='."""
    if "=" not in answer_text:
        return answer_text.strip(), None
    lhs, rhs = answer_text.rsplit("=", 1)
    return lhs.strip(), rhs.strip()


class SafeCountdownEvaluator:
    """Safe evaluator for arithmetic expressions using only +, -, *, / and parentheses.
    - Uses Fractions for exact arithmetic.
    - Tracks integer literals encountered (for number-usage checking).
    """

    ALLOWED_BINOPS = (ast.Add, ast.Sub, ast.Mult, ast.Div)
    ALLOWED_UNARYOPS = (ast.UAdd, ast.USub)

    def __init__(self) -> None:
        self.used_numbers: list[int] = []

    def parse(self, expr: str) -> ast.AST:
        return ast.parse(expr, mode="eval")

    def eval(self, node: ast.AST) -> Fraction:
        return self._eval_node(node)

    def _eval_node(self, node: ast.AST) -> Fraction:
        if isinstance(node, ast.Expression):
            return self._eval_node(node.body)

        if isinstance(node, ast.Constant):
            if isinstance(node.value, bool) or node.value is None:
                raise ValueError("Invalid literal")
            if isinstance(node.value, int):
                self.used_numbers.append(int(node.value))
                return Fraction(int(node.value), 1)
            if isinstance(node.value, float):
                raise ValueError("Float literals not allowed; use integers only")
            raise ValueError("Unsupported literal type")

        if isinstance(node, ast.UnaryOp) and isinstance(node.op, self.ALLOWED_UNARYOPS):
            val = self._eval_node(node.operand)
            return val if isinstance(node.op, ast.UAdd) else -val

        if isinstance(node, ast.BinOp) and isinstance(node.op, self.ALLOWED_BINOPS):
            left = self._eval_node(node.left)
            right = self._eval_node(node.right)

            if isinstance(node.op, ast.Add):
                return left + right
            if isinstance(node.op, ast.Sub):
                return left - right
            if isinstance(node.op, ast.Mult):
                return left * right
            if isinstance(node.op, ast.Div):
                if right == 0:
                    raise ZeroDivisionError("Division by zero")
                return left / right

        raise ValueError(f"Disallowed syntax: {type(node).__name__}")


def eval_expr(expr: str) -> tuple[Fraction, list[int]]:
    ev = SafeCountdownEvaluator()
    tree = ev.parse(expr)
    val = ev.eval(tree)
    return val, ev.used_numbers


def check_number_usage_exactly_once(
    used_nums: list[int], provided_nums: list[int]
) -> bool:
    """Exact rule: LHS must use ALL provided numbers exactly once.
    This is exact multiset equality.
    """
    return Counter(used_nums) == Counter(provided_nums)


def compute_format_reward_func(completions, **kwargs):
    """Format: <think>...</think><answer>...</answer>
    Args:
        completions (list[str]): Generated outputs

    Returns:
          list[float]: Reward scores
    """
    if is_conversational(completions[0]):
        completions = [completion[0]["content"] for completion in completions]

    rewards = []
    for completion in completions:
        try:
            # add synthetic <think> as its already part of the prompt and prefilled for the assistant to more easily match the regex
            completion = "<think>" + completion
            # Check if the format is correct
            regex = r"^<think>([^<]*(?:<(?!/?think>)[^<]*)*)<\/think>\n<answer>([\s\S]*?)<\/answer>$"

            match = re.search(regex, completion, re.DOTALL)
            # if the format is not correct, reward is 0
            if match is None or len(match.groups()) != 2:
                rewards.append(0.0)
            else:
                rewards.append(1.0)
        except Exception:
            rewards.append(0.0)
    return rewards


def compute_numbers_rule_reward(completions, **kwargs):
    """+1 if LHS uses ALL provided nums exactly once. Otherwise 0.
    - Requires exactly one <answer>...</answer>
    - If '=' exists, ignores RHS numbers for usage.
    """
    if is_conversational(completions[0]):
        completions = [completion[0]["content"] for completion in completions]

    num_ls = kwargs["nums"]

    rewards = []
    for completion, provided_numbers in zip(completions, num_ls):
        try:
            ans = extract_final_answer_text(completion)
            if ans is None:
                rewards.append(0.0)
                continue

            lhs, _rhs = split_lhs_rhs(ans)

            # Evaluate LHS to collect used numbers (AST-safe)
            _lhs_val, used_nums = eval_expr(lhs)

            rewards.append(
                1.0
                if check_number_usage_exactly_once(used_nums, provided_numbers)
                else 0.0
            )

        except Exception:
            rewards.append(0.0)

    return rewards


def compute_correct_equation_reward_func(completions, **kwargs):
    """+1 if equation is correct AND uses ALL numbers exactly once on LHS. Otherwise 0.
    Rules:
      - at least one <answer>...</answer>, extracts the final answer
      - LHS uses all numbers exactly once
      - if '=' present: require LHS == RHS == target (ignore RHS number usage)
      - else: require LHS == target
    """
    if is_conversational(completions[0]):
        completions = [completion[0]["content"] for completion in completions]

    target_ls = kwargs["target"]
    num_ls = kwargs["nums"]
    rewards = []
    for completion, target, nums in zip(completions, target_ls, num_ls):
        try:
            ans = extract_final_answer_text(completion)
            if ans is None:
                rewards.append(0.0)
                continue

            lhs, rhs = split_lhs_rhs(ans)
            lhs_val, used_nums = eval_expr(lhs)
            target_frac = Fraction(int(target), 1)

            if rhs is not None and rhs != "":
                rhs_val, _ = eval_expr(rhs)  # ignore rhs usage
                print(lhs_val, rhs_val, target_frac)
                rewards.append(1.0 if (lhs_val == rhs_val == target_frac) else 0.0)
            else:
                print(lhs_val, target_frac)
                rewards.append(1.0 if (lhs_val == target_frac) else 0.0)

        except Exception:
            rewards.append(0.0)

    return rewards

In [26]:
correct_sample_1 = """We need to find an equation using the numbers 19, 36, 55, and 7
exactly once, with basic arithmetic operations, that equals 65. One possible
combination is 55 + 36 - 19 + 7... </think>
<answer> 55 + 36 - 7 - 19 </answer>"""

correct_sample_2 = """ ... </think>
<answer> 55 + 36 - 7 - 19 </answer>"""

wrong_format = (
    """User: Using the numbers [19, 36, 55, 7], create an equation that equals 65."""
)

wrong_format_2 = """To find the equation that equals 79 using the numbers 95, 78, 6, 88, I'll start by adding 88 and 95:                      
95 + 88 = 183                                                                                                              
Now, let's subtract 104 from 183 to get 79:
183 - 104 = 79
<think> 183 - 104 = 79 </think><think> 183 - 104 = 79 </think><answer> 183 - 104 = 79 </answer>"""

wrong_result1 = """ ... </think>
<answer> 55 + 36 - 7 - 18 </answer>"""

correct_result3 = """ ... </think>
<answer> 55 + 36 - 7 - 19 - 10 + 10 </answer>"""


test_rewards = compute_format_reward_func(
    completions=[
        correct_sample_1,
        correct_sample_2,
        wrong_format,
        wrong_format_2,
        wrong_result1,
    ],
    target=["65", "65", "65", "65", "65"],
    nums=[[19, 36, 55, 7]] * 5,
)
assert test_rewards == [1.0, 1.0, 0.0, 0.0, 1.0], "Reward function is not working"
# test_rewards = equation_reward_func(completions=[correct_sample_1, correct_sample_2, wrong_format, wrong_format_2, wrong_result], target=["65", "65", "65", "65", "65"], nums=[[19, 36, 55, 7]] * 5)
# assert test_rewards == [1.0, 1.0, 0.0, 0.0, 0.0], "Reward function is not working"

In [28]:
test_rewards = compute_numbers_rule_reward(
    completions=[
        correct_sample_1,
        correct_sample_2,
        wrong_format,
        wrong_format_2,
        wrong_result1,
        correct_result3,
    ],
    target=["65", "65", "65", "65", "65", "65"],
    nums=[[19, 36, 55, 7]] * 6,
)
assert test_rewards == [1.0, 1.0, 0.0, 0.0, 0.0, 0.0], "Reward function is not working"

In [ ]:
test_rewards = compute_correct_equation_reward_func(
    completions=[
        correct_sample_1,
        correct_sample_2,
        wrong_format,
        wrong_format_2,
        wrong_result1,
        correct_result3,
    ],
    target=["65", "65", "65", "65", "65", "65"],
    nums=[[19, 36, 55, 7]] * 6,
)
assert test_rewards == [1.0, 1.0, 0.0, 0.0, 0.0, 1.0], "Reward function is not working"